# Risk-factor comparisons

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Compare risk factors across diagnoses and between HC participants in the two existing clusters. Export categorical relative risks and continuous-variable effect sizes.


## Inputs

- `files/inputs/combined_cohorts - Copy.xlsx`: master risk-factor records.
- `files/inputs/variables_across_cohorts.xlsx`: availability and variable-type metadata.
- `files/interm_files/kmeans{coh}.xlsx`: cluster labels IDX=0/1 with participant IDs and diagnosis codes.


Default `coh = [3]`; use `[1, 2]` for discovery. Filenames retain the literal list representation.

## Outputs

- `files/interm_files/rr_cat_{coh}_statanalysis.xlsx`: categorical comparisons and relative risks.
- `files/interm_files/cd_cont_{coh}_statanalysis.xlsx`: continuous comparisons and Cohen’s d.


## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
import cohort_data_utils as ops

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2


In [ ]:
def set_global_plot_style():

    """Apply the figure typography and Seaborn style."""
    sns.set_context("paper")

    plt.rcParams['font.family'] = 'Segoe UI'
    plt.rc('font', size=18)
    plt.rc('axes', titlesize=18)
    plt.rc('axes', labelsize=18)
    plt.rc('xtick', labelsize=15)
    plt.rc('ytick', labelsize=15)
    plt.rc('legend', fontsize=15)
    plt.rc('figure', titlesize=18)

set_global_plot_style()

np.set_printoptions(precision=3, suppress=True)


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)


## 2. Select cohort and risk factors

Select variables available in all chosen cohort metadata rows, then classify them as nominal, ordinal, or continuous/discrete. Stress questionnaire columns are renamed to distinguish them from daily stress trajectories.

In [ ]:
coh = [3]

combined_cohorts = pd.read_excel(INPUT_PATH / 'combined_cohorts - Copy.xlsx')
cohort_one_two = combined_cohorts[combined_cohorts['Cohort'].isin(coh)]

cohort_one_two


In [ ]:
cohort_one_two["CODE"].value_counts()


In [ ]:
cohort_one_two.rename(columns={'Stress T1': 'StressQ T1', 'Stress T2': 'StressQ T2','Stress T3': 'StressQ T3','Stress T4': 'StressQ T4',}, inplace=True)


In [ ]:
stress = cohort_one_two[['StressQ T1', 'StressQ T2', 'StressQ T3', 'StressQ T4', 'CODE']]


In [ ]:
cohort_one_two['Gefahr Missbrauch T4'].value_counts()


In [ ]:
variables = pd.read_excel(INPUT_PATH / 'variables_across_cohorts.xlsx')

filtered_cohort = variables[variables['Cohort'].isin(coh)]

columns_with_true = []
for col in variables.columns:
    if col != 'Cohort' and filtered_cohort[col].isin([True]).all():
        columns_with_true.append(col)

filtered_row = variables[variables['Cohort'] == 'type']

row_series = filtered_row.iloc[0]

cat = [col for col in columns_with_true if row_series[col] == 'nominal'  ]
cont = [col for col in columns_with_true if row_series[col] == 'continuous' or row_series[col] == 'discrete']
ord = [col for col in columns_with_true if row_series[col] == 'ordinal']

cat.remove("CODE")


In [ ]:
gefahr_vars = ['Gefahr Missbrauch T1', 'Gefahr Missbrauch T2', 'Gefahr Missbrauch T3', 'Gefahr Missbrauch T4']

to_remove_cont = ["StressQ T1", "StressQ T2", "StressQ T3", "StressQ T3", "StressQ T4"]

cont = [x for x in cont if x not in to_remove_cont]
cont = cont
ord = [x for x in ord if x not in gefahr_vars]

print(f'Cohort {coh}, categorical:', cat)
print(f'Cohort {coh}, continuous:', cont)
print(f'Cohort {coh}, ordinal:', ord)


### Recode selected factors

Apply the supplied binary mappings for feeding, income, education, birth mode, support, and history variables. Preserve the implemented missing-value behavior: missing psychiatric history becomes 1 under the `else` branch, and missing SLE values can become 1 under `where`. These recodings are not revised here.

In [ ]:
cohort_one_two['Breastfeeding T0'].replace(2, 1, inplace=True)
cohort_one_two['Baby blues'].replace(2, pd.NA, inplace=True)

cohort_one_two["Income"] = cohort_one_two["Income"].replace({1:0, 2:0, 3:1, 4:1, 5:1})

cohort_one_two['Psychiatric history'] = pd.to_numeric(cohort_one_two['Psychiatric history'], errors='coerce')

cohort_one_two['Psychiatric history binary'] = cohort_one_two['Psychiatric history'].apply(lambda x: 0 if x == 0 else 1)

cohort_one_two['SLE: Personal'].where(cohort_one_two['SLE: Personal'] <= 1, 1, inplace=True)
cohort_one_two['SLE: Family'].where(cohort_one_two['SLE: Family'] <= 1, 1, inplace=True)
list_cat = cohort_one_two.columns + ['Psychiatric history binary']

cohort_one_two["Gender of the child"] = cohort_one_two["Gender of the child"].replace(2, pd.NA)
cohort_one_two["SLE: Personal"].value_counts()
cohort_one_two['Highest degree of education'] = cohort_one_two['Highest degree of education'].replace({0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 1})
cohort_one_two['Birth mode'] = cohort_one_two['Birth mode'].replace({1: 0, 2: 1, 3: 1, 4: 1, 5: 1})

if coh == [3]:
    cohort_one_two['Support at home t0'] = cohort_one_two['Support at home t0'].replace({0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 0, 6: 0})
    cohort_one_two['Support at home t4'] = cohort_one_two['Support at home t4'].replace({0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 0, 6: 0})
    cohort_one_two['PMS Severity'] = cohort_one_two['PMS Severity'].replace({2:1})

cohort_one_two["Gender of the child"] = cohort_one_two["Gender of the child"].replace(2, pd.NA)


In [ ]:
cohort_one_two["Psychiatric history"].value_counts()


## 3. Compare diagnoses

The categorical helper uses chi-square tests, retaining tables with at most one expected cell below five, and applies BH correction across retained variables. The continuous helper compares ND/PPD/AS with Kruskal–Wallis and runs Bonferroni Dunn tests after raw omnibus p<0.05; omnibus tests are not corrected across variables.

In [ ]:
cat.remove('Psychiatric history')

cat_ord_columns = ord + cat + ['Psychiatric history binary', 'CODE']
cat_ord_data = cohort_one_two[cat_ord_columns]
cat_ord_data.columns = cat_ord_data.columns.str.replace(' ', '_')
list_of_categorical_data_underscore = list(cat_ord_data.columns)

ops.chi_square_test(cat_ord_data, list_of_categorical_data_underscore)


In [ ]:
cohort_one_two.loc[
    cohort_one_two["ID"] == "BH_RK1711",
    "Ablehnung T1"
]


In [ ]:
list_of_continuous_data = cont + ["CODE"]
combined_three_cont = cohort_one_two[list_of_continuous_data].copy()
combined_three_cont.set_axis(list_of_continuous_data, axis=1)

significant_var = ops.kruskal_wallis_test(combined_three_cont, list_of_continuous_data)
significant_var


In [ ]:
cohort_one_two['CODE']


## 4. Link existing cluster assignments

Display diagnosis counts, inspect missingness, then inner-join risk records with cluster data on ID and diagnosis. IDs are assumed unique. Cluster codes remain 0/1.

In [ ]:
if coh == [1, 2]:
    data_path_full  = OUTPUT_FILES / f'kmeans[1, 2].xlsx'
else:
    data_path_full  = OUTPUT_FILES / f'kmeans[3].xlsx'

full = pd.read_excel(data_path_full)
full = full.rename(columns={'Diagnosis': 'CODE'})
full = full.rename(columns={'Cluster': 'IDX'})


In [ ]:
full['IDX'].value_counts(

)


In [ ]:
col_names = full.columns.tolist()

full['Diagnosis'] = ''

full.loc[full['CODE'] == 'ND', 'Diagnosis'] = 'HC'
full.loc[full['CODE'] == 'AS', 'Diagnosis'] = 'AD'

full.loc[full['Diagnosis'] == '', 'Diagnosis'] = full['CODE']

cohort_one_two['Diagnosis'] = ''

cohort_one_two.loc[cohort_one_two['CODE'] == 'ND', 'Diagnosis'] = 'HC'
cohort_one_two.loc[cohort_one_two['CODE'] == 'AS', 'Diagnosis'] = 'AD'

cohort_one_two.loc[cohort_one_two['Diagnosis'] == '', 'Diagnosis'] = cohort_one_two['CODE']

custom_palette = {'PPD': '#E2725B',
                'HC': '#4682B4',
                'AD': '#6B8E23'}


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), sharey=True)

for idx, ax in zip(full['IDX'].unique(), axes.flatten()):
    sns.countplot(data=full[full['IDX'] == idx], x='Diagnosis', palette=custom_palette, ax=ax)
    ax.set_title(f'Group {idx}')
    ax.set_xlabel('Diagnosis')
    ax.set_ylabel('Count')
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()
full.groupby('IDX')['CODE'].value_counts()


In [ ]:
full.columns


In [ ]:
all = cat + cont + ord + ["CODE"]
df = cohort_one_two[all]

missing_values = df.isna().sum()

missing_per_diagnosis = df.groupby('CODE').apply(lambda x: x.isna().sum())

print("Total Missing Values per Column:")
print(missing_values.to_string())


In [ ]:
full_df = pd.merge(cohort_one_two, full, on=['ID', 'Diagnosis'], how='inner')
full_df['SLE: Family'].unique()
full_df["Psychiatric history binary"].value_counts()


In [ ]:
cat_ord_columns


In [ ]:
pd.options.mode.chained_assignment = None
cat_ord_columns = ord + cat + ['Psychiatric history binary']

combined_one_two_cat = full_df[cat_ord_columns]
combined_one_two_cat['IDX'] = full_df['IDX']
combined_one_two_cat['Diagnosis'] = full_df['Diagnosis']
from scikit_posthocs import posthoc_dunn
import scipy.stats as stats

combined_one_two_cont = full_df[cont].copy()
combined_one_two_cont['Diagnosis'] = full_df['Diagnosis']
combined_one_two_cont['IDX'] = full_df['IDX']
combined_one_two_cont.columns

combined_one_two_cont


In [ ]:
list_of_continuous_data


In [ ]:
cohort_one_two['in vitro fertilization2'].value_counts()


### Select variables for cluster comparisons

Retain the listed categorical/ordinal factors and remove the explicitly excluded variables. List removals assume those names are present and some lists are shared by reference.

In [ ]:
if coh == [3]:
    continuous_var = cont

categ_var = cat_ord_columns

categ_var.remove('Development of the child T4')


In [ ]:
categ_var


In [ ]:
remove_values = [
    "Completed professional education",
    "SLE: Personal",
    "SLE: Family",
    "Iron supplementation",
    "Family status",
    "Eisenpräparat",
    "In vivo fertilization"
]

for val in remove_values:
    if val in categ_var:
        categ_var.remove(val)
        print(f"Removed: {val}")

print("\nFinal list:")
print(categ_var)


## 5. Categorical factors: HC Cluster 1 versus HC Cluster 2

Compare HC at IDX=1 (exposed) with HC at IDX=0 (control); events are values equal to 1. Relative risk is the event proportion at IDX=1 divided by that at IDX=0, with SciPy’s 95% interval. Use chi-square when the helper’s expected-count screen passes, otherwise Fisher’s exact test. Missing binary rows/columns are skipped only where handled. No multiple-testing correction is applied to this export; stars use p-values already rounded to three decimals. These are factor-prevalence comparisons, not incident-disease predictions.

In [ ]:
from scipy.stats import fisher_exact
from scipy.stats.contingency import relative_risk
from scipy.stats.contingency import odds_ratio

selected = combined_one_two_cat[categ_var]
selected['IDX'] = combined_one_two_cat['IDX']
selected['Diagnosis'] = combined_one_two_cat['Diagnosis']
for col in categ_var:
    selected[col] = selected[col].replace({0: 0.0, 1: 1.0})

var0 = 'HC'
var1 = 'HC'

print(f'Compare {var0} from Group 0 with {var1} from Group 1')
filter_idx_0 = selected[(selected['IDX'] == 0) & (selected['Diagnosis'] == var0)]
filter_idx_1_ppd = selected[(selected['IDX'] == 1) & (selected['Diagnosis'] == var1)]

result = pd.concat([filter_idx_0, filter_idx_1_ppd])

p_values = []
contingency_tables = []
validities = []
rr_values = []

summary_data = []

for var in categ_var:
    selected_df = result[result['IDX'].isin([0, 1])]
    contingency_table = pd.crosstab(selected_df[var], selected_df['IDX'])

    exposed_total = contingency_table[1].sum()
    control_total = contingency_table[0].sum()

    exposed_cases = (filter_idx_1_ppd[var] == 1).sum()
    control_cases = (filter_idx_0[var] == 1).sum()
    pass

    try:
        a = contingency_table.loc[1.0, 1]
        b = contingency_table.loc[1.0, 0]
        c = contingency_table.loc[0.0, 1]
        d = contingency_table.loc[0.0, 0]
    except KeyError:
        print(f"Issue in contingency table for variable '{var}' — skipping.")
        continue

    odds_table = [[a, b],
                [c, d]]

    is_valid, chi2, p, dof, expected = ops.check_contingency_table(odds_table)
    if is_valid:
        validities.append(is_valid)
        p_values.append(p)
    else:
        stat, p = fisher_exact(odds_table)
        p_values.append(p)

    rr = relative_risk(
        exposed_cases=exposed_cases,
        exposed_total=exposed_total,
        control_cases=control_cases,
        control_total=control_total
    )
    odds = odds_ratio(odds_table)

    summary_data.append({
    'Variable': var,
    'Exposed population': exposed_total,
    'Exposed events': exposed_cases,
    'Unexposed population': control_total,
    'Unexposed events': control_cases,
    'Validity': is_valid,
    'P-value': round(p,3),
    'RR': round(rr.relative_risk,3),
    'CI Lower': round(rr.confidence_interval(confidence_level=0.95).low,3),
    'CI Upper': round(rr.confidence_interval(confidence_level=0.95).high,3)

    })
summary_df = pd.DataFrame(summary_data)

def significance_marker(row):
    """Mark rounded, unadjusted p-values below 0.05."""
    if row['P-value'] < 0.05:
        return '*'
    else:
        return ''

summary_df['Significance'] = summary_df.apply(significance_marker, axis=1)
summary_df.to_excel(OUTPUT_FILES / f'rr_cat_{coh}_statanalysis.xlsx', index=False)

summary_df


In [ ]:
cont


In [ ]:
continuous_var = cont
selected_cont = combined_one_two_cont[continuous_var]
selected_cont['IDX'] = combined_one_two_cat['IDX']
selected_cont['Diagnosis'] = combined_one_two_cat['Diagnosis']


In [ ]:
selected_cont


In [ ]:
pd.options.mode.chained_assignment = None
cat_ord_columns = ord + cat + ['Psychiatric history binary']

combined_one_two_cont = full_df[cat_ord_columns]
combined_one_two_cont['IDX'] = full_df['IDX']
combined_one_two_cont['Diagnosis'] = full_df['Diagnosis']
from scikit_posthocs import posthoc_dunn
import scipy.stats as stats

combined_one_two_cont = full_df[cont].copy()
combined_one_two_cont['Diagnosis'] = full_df['Diagnosis']
combined_one_two_cont['IDX'] = full_df['IDX']
combined_one_two_cont.columns

combined_one_two_cont


In [ ]:
var1
continuous_var.remove('HAMILTON')
if coh == [3]:
    continuous_var.remove('Weight before pregnancy (kg)')
    continuous_var.remove('Weight after birth')
    continuous_var.remove('Weight 3 months after birth')


## 6. Continuous factors

Cohen’s d is `(mean of group 2 − mean of group 1) / pooled sample SD`. Independently resample observations within each group 10,000 times, resetting a NumPy generator to seed 42 for each call, and use the 2.5th–97.5th percentiles for the interval. Empty groups, insufficient nonmissing data, and zero pooled variance are not fully guarded against.

In [ ]:
def cohen_d_bootstrap(group1, group2, n_bootstrap = 10000, alpha=0.05, random_state=42):

    """Return group-2 minus group-1 Cohen’s d and a percentile bootstrap interval."""
    rng = np.random.default_rng(random_state)

    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    pooled_std = np.sqrt(((len(group1) - 1)*std1**2 + (len(group2) - 1)*std2**2) / (len(group1) + len(group2) - 2))
    # Positive values indicate a higher mean in group 2.
    mean_diff = mean2 - mean1
    cohen_d = mean_diff / pooled_std

    boot_d = []
    for _ in range(n_bootstrap):
        sample1 = rng.choice(group1, size=len(group1), replace=True)
        sample2 = rng.choice(group2, size=len(group2), replace=True)
        m1, m2 = np.mean(sample1), np.mean(sample2)
        s1, s2 = np.std(sample1, ddof=1), np.std(sample2, ddof=1)

        pooled = np.sqrt(((len(sample1) - 1)*s1**2 + (len(sample2) - 1)*s2**2) / (len(sample1) + len(sample2) - 2))
        d = (m2 - m1) / pooled
        boot_d.append(d)

    ci_lower = np.percentile(boot_d, 100 * alpha / 2)

    ci_upper = np.percentile(boot_d, 100 * (1 - alpha / 2))

    return cohen_d, ci_lower, ci_upper


In [ ]:
continuous_var


In [ ]:
var0


Coerce each variable to numeric and remove missing scores within groups. Use two-group Kruskal–Wallis tests and export Cohen’s d with bootstrap intervals. The minimum-size check uses pre-drop row counts. FDR code is inactive; stars use the rounded, unadjusted p-values.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

print(f'Compare {var0} from Group 0 with {var1} from Group 1')

filter_idx_0_cont = selected_cont[(selected_cont['IDX'] == 0) & (selected_cont['Diagnosis'] == var0)]
filter_idx_1_ppd_cont = selected_cont[(selected_cont['IDX'] == 1) & (selected_cont['Diagnosis'] == var1)]

result_cont = pd.concat([filter_idx_0_cont, filter_idx_1_ppd_cont])

significant_var = []
p_values = []
cohens_d_values = []

summary_data_cont = []

for var in continuous_var:

    filter_idx_0_cont[var] = pd.to_numeric(filter_idx_0_cont[var], errors='coerce')
    filter_idx_1_ppd_cont[var] = pd.to_numeric(filter_idx_1_ppd_cont[var], errors='coerce')

    data1 = filter_idx_0_cont[var].dropna()
    data2 = filter_idx_1_ppd_cont[var].dropna()

    print(var)

    if len(filter_idx_0_cont[var]) < 2 or len(filter_idx_1_ppd_cont[var]) < 2:
        print(f"Skipping {var} due to insufficient data")
        continue

    Kruskal_result = stats.kruskal(data1,
                                data2)
    print('p:', Kruskal_result.pvalue)
    p_values.append(Kruskal_result.pvalue)

    cohens_d, ci_lower, ci_upper = cohen_d_bootstrap(data1, data2)

    h = filter_idx_0_cont[var].count()
    hr = filter_idx_1_ppd_cont[var].count()

    summary_data_cont.append({
        'Variable': var,
        'Cohen\'s d': round(cohens_d,3),
        'CI Lower': round(ci_lower,3),
        'CI Upper': round(ci_upper,3),
        'P-value': round(Kruskal_result.pvalue,3),
        'Healthy': h,
        'Healthy at risk': hr,

    })

valid_p_values = [p for p in p_values if not np.isnan(p)]
valid_indices = [i for i, p in enumerate(p_values) if not np.isnan(p)]

summary_df_cont = pd.DataFrame(summary_data_cont)
summary_df_cont['P-value'] = pd.to_numeric(summary_df_cont['P-value'], errors='coerce')
def significance_marker(row):
    """Mark rounded, unadjusted p-values below 0.05."""
    if row['P-value'] < 0.05:
        return '*'
    else:
        return ''

summary_df_cont['Significance'] = summary_df_cont.apply(significance_marker, axis=1)
summary_df_cont.to_excel(OUTPUT_FILES / f'cd_cont_{coh}_statanalysis.xlsx', index=False)

summary_df_cont


In [ ]:
OUTPUT_FILES
